# ARIMA & Seasonal ARIMA Forecasting
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/03_Machine_Learning/projects/forecasting/arima_forecasting.ipynb)

ARIMA models a series from its own past values (AR), differenced to remove trend (I), plus past forecast errors (MA). SARIMA adds seasonal terms - the classical baseline every ML forecaster must know.

Built on `statsmodels`; synthetic monthly data with trend + seasonality so the truth is known.

## 1. Create and inspect the series

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

rng = np.random.default_rng(42)
idx = pd.date_range("2018-01-01", periods=84, freq="MS")
t = np.arange(84)
y = pd.Series(50 + 0.6 * t                                   # trend
              + 12 * np.sin(2 * np.pi * t / 12)              # yearly season
              + rng.normal(0, 3, 84), index=idx, name="sales")

y.plot(figsize=(11, 3.5), title="Monthly sales: trend + season + noise")
plt.show()

## 2. Stationarity check + differencing

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf(name, s):
    p = adfuller(s)[1]
    print(f"{name:<14} ADF p-value = {p:.4f}  {'stationary' if p < 0.05 else 'NON-stationary'}")

adf("raw", y)
adf("differenced d1", y.diff().dropna())
adf("d1 + seasonal D1", y.diff().diff(12).dropna())

Rule: p < 0.05 => stationary => that differencing level is enough.

## 3. ACF/PACF guide the orders

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
d = y.diff().diff(12).dropna()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
plot_acf(d, lags=36, ax=axes[0]); plot_pacf(d, lags=36, method="ywm", ax=axes[1])
plt.tight_layout(); plt.show()

## 4. Fit SARIMA + walk-forward evaluation

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

train, test = y[:-12], y[-12:]
model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)).fit(disp=False)
pred = model.get_forecast(12).predicted_mean

from sklearn.metrics import mean_absolute_error, mean_squared_error
rmse = mean_squared_error(test, pred) ** 0.5
mape = (abs(test - pred) / test).mean() * 100
print(f"RMSE={rmse:.2f}  MAPE={mape:.1f}%")

ax = y.plot(figsize=(11, 4), label="actual")
pred.plot(ax=ax, style="--o", color="crimson", label="SARIMA forecast")
ax.legend(); ax.set_title("Held-out year forecast"); plt.show()

## Takeaways
- Workflow: visualize -> make stationary (ADF) -> read ACF/PACF -> fit -> validate on hold-out tail.
- `(p,d,q)x(P,D,Q,s)` with `s=12` monthly seasonality covers most business series.
- Automate order search with `pmdarima.auto_arima`; compare against naive seasonal benchmark always.
- Prophet notebook next shows the decomposable/regression alternative.